In [1]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2020

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2020 folder created successfully.
2020_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5001  2020-01-10
1  5002  2020-01-20
2  5001  2020-01-20
3  5011  2020-01-21
4  5002  2020-01-22

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 7

Missing Temperature Values:
Tmax    0
Tmin    0
RH      0
dtype: int64
2020_cleaned_temp.csv created successfully.
2020_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5001        -1 2020-01-10  29.66  15.59   
1       111                 SCPU  5002        -1 2020-01-20  32.04  16.56   
2       111                 SCPU  5001        -1 2020-01-20  32.04  16.56   
3       111                 SCPU  5011        -1 2020-01-21  25.31  22.82   
4       111                 SCP

In [2]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2021

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2021 folder created successfully.
2021_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5001  2021-01-15
1  5020  2021-02-10
2  5002  2021-02-16
3  5063  2021-02-24
4  5064  2021-03-10

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 8

Missing Temperature Values:
Tmax    0
Tmin    0
RH      0
dtype: int64
2021_cleaned_temp.csv created successfully.
2021_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5001        -1 2021-01-15  32.57  19.17   
1       111                 SCPU  5020        -5 2021-02-10  29.95  11.62   
2       111                 SCPU  5002        -1 2021-02-16  31.69  18.12   
3       111                 SCPU  5063        -1 2021-02-24  34.92  15.67   
4       111                 SCP

In [3]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2022

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2022 folder created successfully.
2022_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5042  2022-06-13
1  5042  2022-06-13
2  5042  2022-06-13
3  5042  2022-06-13
4  5042  2022-06-13

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 6

Missing Temperature Values:
Tmax    0
Tmin    0
RH      0
dtype: int64
2022_cleaned_temp.csv created successfully.
2022_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
1       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
2       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
3       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
4       111                 SCP

In [4]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2022

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2022 folder created successfully.
2022_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5042  2022-06-13
1  5042  2022-06-13
2  5042  2022-06-13
3  5042  2022-06-13
4  5042  2022-06-13

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 6

Missing Temperature Values:
Tmax    0
Tmin    0
RH      0
dtype: int64
2022_cleaned_temp.csv created successfully.
2022_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
1       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
2       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
3       111                 SCPU  5042        -1 2022-06-13  33.21  24.54   
4       111                 SCP

In [5]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2023

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2023 folder created successfully.
2023_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5030  2023-01-24
1  5042  2023-01-31
2  5021  2023-06-19
3  5021  2023-06-19
4  5021  2023-06-19

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 4

Missing Temperature Values:
Tmax    0
Tmin    0
RH      0
dtype: int64
2023_cleaned_temp.csv created successfully.
2023_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5030        -1 2023-01-24  22.22  12.59   
1       111                 SCPU  5042        -1 2023-01-31  30.63  16.99   
2       111                 SCPU  5021        -1 2023-06-19  38.16  30.47   
3       111                 SCPU  5021        -1 2023-06-19  38.16  30.47   
4       111                 SCP

In [6]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2024

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2024 folder created successfully.
2024_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5001  2024-04-27
1  5001  2024-04-27
2  5001  2024-04-27
3  5001  2024-06-06
4  5033  2024-06-26

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 14

Missing Temperature Values:
Tmax    9
Tmin    9
RH      9
dtype: int64
2024_cleaned_temp.csv created successfully.
2024_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5001        -1 2024-04-27  43.04  26.64   
1       111                 SCPU  5001        -1 2024-04-27  43.04  26.64   
2       111                 SCPU  5001        -1 2024-04-27  43.04  26.64   
3       111                 SCPU  5001        -1 2024-06-06  36.18  27.94   
4       111                 SC

In [7]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2025

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2025 folder created successfully.
2025_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5001  2025-01-04
1     0  2025-01-06
2     0  2025-01-06
3     0  2025-01-06
4     0  2025-01-06

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 41

Missing Temperature Values:
Tmax    6
Tmin    6
RH      6
dtype: int64
2025_cleaned_temp.csv created successfully.
2025_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5001        -1 2025-01-04  32.64  15.09   
1       111                 SCPU     0        -1 2025-01-06  30.31  12.80   
2       111                 SCPU     0        -1 2025-01-06  30.31  12.80   
3       111                 SCPU     0        -1 2025-01-06  30.31  12.80   
4       111                 SC

In [8]:
import pandas as pd
import os

# ============================================================
# DEFINE YEAR
# ============================================================

YEAR = 2026

# ============================================================
# DEFINE BASE PATH
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\111"

# ============================================================
# CREATE YEAR FOLDER
# ============================================================

year_folder = os.path.join(base_path, str(YEAR))

os.makedirs(year_folder, exist_ok=True)

print(f"{YEAR} folder created successfully.")

# ============================================================
# READ ORIGINAL PCB DATASET
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "111.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# FILTER YEAR-WISE DATA
# ============================================================

df = df[
    df['Pstng Date'].dt.year == YEAR
]

# ============================================================
# RESET INDEX
# ============================================================

df = df.reset_index(drop=True)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned.csv"
)

if os.path.exists(cleaned_path):
    os.remove(cleaned_path)

df.to_csv(cleaned_path, index=False)

print(f"{YEAR}_cleaned.csv created successfully.")

# ============================================================
# READ TEMPERATURE DATASET
# ============================================================

temperature_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"
)

# ============================================================
# CLEAN TEMPERATURE DATASET
# ============================================================

temperature_df.columns = temperature_df.columns.str.strip()

# ============================================================
# CONVERT DATE COLUMNS
# ============================================================

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# CONVERT DATES TO SAME FORMAT
# ============================================================

temperature_df['Date'] = (
    temperature_df['Date']
    .dt.strftime('%Y-%m-%d')
)

df['Pstng Date'] = (
    df['Pstng Date']
    .dt.strftime('%Y-%m-%d')
)

# ============================================================
# STANDARDIZE SLOC COLUMNS
# ============================================================

df['SLoc'] = (
    pd.to_numeric(df['SLoc'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

temperature_df['SLOC'] = (
    pd.to_numeric(temperature_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# DEBUG CHECK
# ============================================================

print("\nPCB Dataset Sample:")
print(df[['SLoc', 'Pstng Date']].head())

print("\nTemperature Dataset Sample:")
print(temperature_df[['SLOC', 'Date']].head())

# ============================================================
# CHECK COMMON SLOC COUNT
# ============================================================

common_slocs = set(df['SLoc']).intersection(
    set(temperature_df['SLOC'])
)

print(f"\nCommon SLOC Count: {len(common_slocs)}")

# ============================================================
# MERGE TEMPERATURE DATA
# ============================================================

merged_df = pd.merge(

    df,

    temperature_df[[
        'SLOC',
        'Date',
        'Tmax',
        'Tmin',
        'RH',
        'Delta_T'
    ]],

    left_on=['SLoc', 'Pstng Date'],
    right_on=['SLOC', 'Date'],

    how='left'
)

# ============================================================
# REMOVE EXTRA COLUMNS
# ============================================================

merged_df.drop(
    columns=['SLOC', 'Date'],
    inplace=True
)

# ============================================================
# CHECK NULL VALUES
# ============================================================

print("\nMissing Temperature Values:")

print(
    merged_df[['Tmax', 'Tmin', 'RH']]
    .isnull()
    .sum()
)

# ============================================================
# SAVE TEMPERATURE MERGED DATASET
# ============================================================

temp_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp.csv"
)

if os.path.exists(temp_path):
    os.remove(temp_path)

merged_df.to_csv(temp_path, index=False)

print(f"{YEAR}_cleaned_temp.csv created successfully.")

# ============================================================
# READ CITIES DATASET
# ============================================================

cities_df = pd.read_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"
)

# ============================================================
# CLEAN CITIES DATASET
# ============================================================

cities_df.columns = cities_df.columns.str.strip()

cities_df['SLOC'] = (
    pd.to_numeric(cities_df['SLOC'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# ============================================================
# CONVERT DATE COLUMN BACK TO DATETIME
# ============================================================

merged_df['Pstng Date'] = pd.to_datetime(
    merged_df['Pstng Date']
)

# ============================================================
# CREATE MONTH COLUMN
# ============================================================

merged_df['Month'] = (
    merged_df['Pstng Date']
    .dt.month
)

# ============================================================
# CREATE SEASON COLUMN
# ============================================================

def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'

merged_df['Season'] = (
    merged_df['Month']
    .apply(get_season)
)

# ============================================================
# MERGE REGION COLUMN
# ============================================================

merged_df = pd.merge(

    merged_df,

    cities_df[['SLOC', 'Region']],

    left_on='SLoc',
    right_on='SLOC',

    how='left'
)

# ============================================================
# REMOVE EXTRA SLOC COLUMN
# ============================================================

merged_df.drop(
    columns=['SLOC'],
    inplace=True
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_path = os.path.join(
    year_folder,
    f"{YEAR}_cleaned_temp_season_n_region.csv"
)

if os.path.exists(final_path):
    os.remove(final_path)

merged_df.to_csv(final_path, index=False)

print(f"{YEAR}_cleaned_temp_season_n_region.csv created successfully.")

# ============================================================
# DISPLAY FINAL DATA
# ============================================================

print("\nFinal Dataset Sample:")

print(merged_df.head())

2026 folder created successfully.
2026_cleaned.csv created successfully.

PCB Dataset Sample:
   SLoc  Pstng Date
0  5043  2026-01-01
1  5043  2026-01-01
2  5030  2026-01-03
3  5030  2026-01-03
4  5030  2026-01-03

Temperature Dataset Sample:
   SLOC        Date
0  5015  2020-01-01
1  5015  2020-01-02
2  5015  2020-01-03
3  5015  2020-01-04
4  5015  2020-01-05

Common SLOC Count: 35

Missing Temperature Values:
Tmax    0
Tmin    0
RH      0
dtype: int64
2026_cleaned_temp.csv created successfully.
2026_cleaned_temp_season_n_region.csv created successfully.

Final Dataset Sample:
   Material Material Description  SLoc  Quantity Pstng Date   Tmax   Tmin  \
0       111                 SCPU  5043        -1 2026-01-01  27.36  22.93   
1       111                 SCPU  5043         1 2026-01-01  27.36  22.93   
2       111                 SCPU  5030        -1 2026-01-03  18.40   5.11   
3       111                 SCPU  5030        -1 2026-01-03  18.40   5.11   
4       111                 SC